# Прављење апликација за генерисање слика (OpenAI)

Велики језички модели не служе само за генерисање текста. Такође можете генерисати слике из текстуалних описа. Слике као модалитет су корисне у МедТеку, архитектури, туризму, развоју игара, маркетингу и још много тога. У овој лекцији радимо са данашњим **GPT Image** моделима на OpenAI платформи.

## Циљеви учења

На крају ове лекције моћи ћете да:

- Објасните шта је генерисање слика и где је корисно.
- Разумете породицу модела `gpt-image` и како се разликује од заоставштина DALL·E модела.
- Направите апликацију за генерисање слика и уређујете слике.

## Шта је генерисање слика?

Модели за генерисање слика праве слике из текстуалног упита. Модерни модели као што је `gpt-image` уче односе између текста и слика током тренинга, а затим итеративно претварају случајни шум у слику која одговара вашем упиту.

Две познате породице модела слика су:

- **`gpt-image` (OpenAI)** — тренутна генерација која се користи у овој лекцији. Подржава генерисање слика из текста и уређивање слика (инпејтинг са маском).
- **Midjourney** — популаран модел треће стране са сопственом услугом и радним током заснованим на Discord-у.

> Старији OpenAI модели за слике — **DALL·E 2** и **DALL·E 3** — су заоставштина. DALL·E 3 више није доступан за нове распоредоме, а функције као што је `create_variation` су постојале само у DALL·E 2. За нове апликације користите `gpt-image` моделе.

> **Важно:** `gpt-image` модели враћају генерисану слику као **base64** (`b64_json`), а не као URL. Ваш код декодира base64 низ у бајтове и чува слику — нема URL за скидање слике.


## Креирање ваше прве апликације за генерисање слика

Шта је потребно да се направи апликација за генерисање слика? Потребне су вам следеће библиотеке:

- **python-dotenv**, ова библиотека вам је топло препоручена да бисте чували своје тајне у *.env* фајлу одвојено од кода.
- **openai**, ову библиотеку ћете користити за интеракцију са OpenAI API-јем.
- **pillow**, за рад са сликама у Python-у.


1. Направите фајл *.env* са следећим садржајем:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Прикупите горе наведене библиотеке у фајл под именом *requirements.txt* на следећи начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затим, креирајте виртуелно окружење и инсталирајте библиотеке:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows, користите следеће команде за креирање и активирање вашег виртуелног окружења:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додајте следећи код у фајл назван *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # увезите dotenv
    dotenv.load_dotenv()

    # Креирајте OpenAI објекат (учитава OPENAI_API_KEY из вашег .env)
    client = openai.OpenAI()


    try:
        # Креирајте слику користећи API за генерисање слика
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Унесите ваш текст упита овде
            size='1024x1024',
            n=1
        )
        # Поставите директоријум за сачувану слику
        image_dir = os.path.join(os.curdir, 'images')

        # Ако директоријум не постоји, креирајте га
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Иницијализујте путању слике (обратите пажњу да тип фајла треба бити png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image модели враћају слику као base64 (b64_json), не као URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Прикажи слику у подразумеваном прегледачу слика
        image = Image.open(image_path)
        image.show()

    # ухватите изузетке
    except openai.BadRequestError as err:
        print(err)

    ```

Хајде да објаснимо овај код:

- Прво, увозимо библиотеке које су нам потребне, укључујући OpenAI библиотеку, dotenv библиотеку, base64 модул и Pillow библиотеку.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Након тога, креирамо клијента, који чита API кључ из вашег ``.env``.

    ```python
    # Креирајте OpenAI објекат
    client = openai.OpenAI()
    ```

- Следеће, генеришемо слику:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Унесите ваш текст упита овде
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` модели враћају слику као **base64** стринг у `data[0].b64_json`. Ми га декодујемо у бајтове и уписујемо у фајл — нема URL адресу за преузимање.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- На крају, отварамо слику и користимо стандардни прегледач слика да је прикажемо:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Више детаља о генерисању слике

Хајде да погледамо параметре функције `images.generate`:

- **model**, је модел слике који се користи (на пример `gpt-image-1`).
- **prompt**, је текстуални опис који се користи за генерисање слике. Овде је "Зека на коњу, држи лизалицу, на магловитој ливади где расту нарциси".
- **size**, је величина генерисане слике (на пример `1024x1024`, `1536x1024`, `1024x1536`, или `"auto"`).
- **n**, је број генерисаних слика. Овде генеришемо једну.

> Модели за слике не користе параметар `temperature` — то је контролa за генерисање текста. Да бисте добили разноликост, поново позовите API; да бисте смањили разноликост, направите ваш опис прецизнијим.

## Додатне могућности генерисања слика

Видели сте како се слика генерише са неколико линија Питхон кода. `gpt-image` модели такође могу **уредити** постојећу слику. Пружањем слике, опционо **маске** (која означава област за промену) и описа, можете изменити део слике — на пример, додати шешир нашем зеци.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# измене се такође враћају као base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основна слика садржи само зеца; коначна слика има шешир на зецу.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
